# OCR via macOS Vision (ocrmac)

Uses Apple's built-in Vision framework — the same engine Preview.app uses for text selection in images. Runs entirely locally on your M4. **No API key. No rate limits. No cost.**

Expected runtime: ~14-20 min for 1,397 PDFs.

## Setup (one time)

```bash
conda activate ML
pip install ocrmac
```

## Run

1. Make sure Drive Desktop is running and the `litigation-pipeline/pdfs/` folder is local.
2. Run cells 1 → 5 top to bottom.
3. Outputs land in `extracted/` and auto-sync to Drive within ~1 min.

## Quality note

macOS Vision is excellent on **printed** text (SC-100 form fields, headers, body) and decent on **handwritten** text (less reliable but usually readable). It's not an LLM — there's no context understanding — just very good text recognition. The downstream feature-extraction step (gpt-4o-mini reading the .txt) will do the contextual work.

In [2]:
from pathlib import Path

DRIVE_DIR = Path(
    "/Users/ernesto/Library/CloudStorage/"
    "GoogleDrive-ernestod1998@gmail.com/My Drive/litigation-pipeline"
)
PDF_DIR = DRIVE_DIR / "pdfs"
OUTPUT_DIR = DRIVE_DIR / "extracted"

# Set to small int (e.g. 5) to test on a handful before committing.
LIMIT = None

DOC_TYPE_WHITELIST = {
    "CLAIM_OF_PLAINTIFF",
    "DEFENDANT_S_CLAIM",
    "JUDGMENT",
    "ORDER",
    "DISMISSAL",
    "Notice_of_Entry_of_Judgment",
    "DECLARATION_OF_APPEARANCE",
    "STIPULATION",
    "COURT_JUDGMENT",
}


def is_doc_type_wanted(description: str) -> bool:
    desc_upper = description.upper()
    return any(w.upper() in desc_upper for w in DOC_TYPE_WHITELIST)


def _doc_type(p):
    return p.stem.split("_", 1)[-1] if "_" in p.stem else p.stem


assert PDF_DIR.exists(), f"PDF_DIR not found: {PDF_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_pdfs = sorted(PDF_DIR.glob("*.pdf"))
whitelisted = [p for p in all_pdfs if is_doc_type_wanted(_doc_type(p))]
needs_ocr = [p for p in whitelisted if not (OUTPUT_DIR / f"{p.stem}.txt").exists()]

if LIMIT:
    needs_ocr = needs_ocr[:LIMIT]

print(f"PDFs total:            {len(all_pdfs)}")
print(f"Whitelisted:           {len(whitelisted)}")
print(f"Already extracted:     {len(whitelisted) - len(needs_ocr) if not LIMIT else '(filtered)'}")
print(f"To OCR this run:       {len(needs_ocr)}")

PDFs total:            8732
Whitelisted:           8617
Already extracted:     8615
To OCR this run:       2


In [3]:
# Import ocrmac (Apple Vision wrapper). If this fails, run: pip install ocrmac
import io
import fitz
from PIL import Image
from ocrmac import ocrmac

DPI = 200  # Higher than the 150 used for cloud OCR — Apple Vision benefits from more pixels


def page_to_pil(page, dpi=DPI):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    return Image.open(io.BytesIO(pix.tobytes("png")))


def ocr_page(pil_image):
    """Run Apple Vision OCR on a PIL image. Returns reconstructed text in reading order."""
    # recognition_level="accurate" is slower but much better quality than "fast".
    # language_preference=["en-US"] tells Vision to optimize for English.
    result = ocrmac.OCR(
        pil_image,
        recognition_level="accurate",
        language_preference=["en-US"],
    ).recognize()
    # result is list of (text, confidence, bbox). Sort by vertical position (top→bottom),
    # then horizontal (left→right) for natural reading order.
    sorted_lines = sorted(result, key=lambda r: (-r[2][1], r[2][0]))
    return "\n".join(text for text, _conf, _box in sorted_lines)


def ocr_pdf(pdf_path):
    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        return None, f"open failed: {e}"

    pages_text = []
    try:
        for page in doc:
            img = page_to_pil(page)
            text = ocr_page(img)
            pages_text.append(text)
    except Exception as e:
        doc.close()
        return None, f"OCR failed: {e}"

    doc.close()
    return "\n\n--- PAGE BREAK ---\n\n".join(pages_text), None


print("ocrmac loaded. Vision framework ready.")

ocrmac loaded. Vision framework ready.


In [4]:
import time

t0 = time.time()
ok = 0
errors = []
n = len(needs_ocr)

for i, pdf_path in enumerate(needs_ocr, start=1):
    txt_path = OUTPUT_DIR / f"{pdf_path.stem}.txt"
    if txt_path.exists():
        continue

    text, err = ocr_pdf(pdf_path)
    if err:
        errors.append((pdf_path.name, err))
        print(f"[{i}/{n}] FAIL {pdf_path.name}: {err}")
        continue

    txt_path.write_text(text, encoding="utf-8")
    ok += 1
    kb = txt_path.stat().st_size / 1024
    if i % 10 == 0 or i <= 5:
        rate = i / (time.time() - t0)
        eta_min = (n - i) / rate / 60
        print(f"[{i}/{n}] {pdf_path.name} ({kb:.1f} KB) — {rate:.1f} PDFs/sec, ETA {eta_min:.0f} min")

elapsed = time.time() - t0
print(f"\n{'='*60}")
print(f"Done in {elapsed/60:.1f} min")
print(f"  OK:     {ok}")
print(f"  Errors: {len(errors)}")
if errors:
    print("\nFirst 10 errors:")
    for name, err in errors[:10]:
        print(f"  {name}: {err}")

[1/2] FAIL CSM24868935_CLAIM_OF_PLAINTIFF.pdf: OCR failed: code=7: Invalid number of pages
[2/2] FAIL CSM25870166_Notice_of_Entry_of_Judgment.pdf: open failed: Failed to open file '/Users/ernesto/Library/CloudStorage/GoogleDrive-ernestod1998@gmail.com/My Drive/litigation-pipeline/pdfs/CSM25870166_Notice_of_Entry_of_Judgment.pdf'.

Done in 0.0 min
  OK:     0
  Errors: 2

First 10 errors:
  CSM24868935_CLAIM_OF_PLAINTIFF.pdf: OCR failed: code=7: Invalid number of pages
  CSM25870166_Notice_of_Entry_of_Judgment.pdf: open failed: Failed to open file '/Users/ernesto/Library/CloudStorage/GoogleDrive-ernestod1998@gmail.com/My Drive/litigation-pipeline/pdfs/CSM25870166_Notice_of_Entry_of_Judgment.pdf'.


In [5]:
# Summary + spot-check the most recently-OCR'd CSM24 claim file.
all_txts = sorted(OUTPUT_DIR.glob("*.txt"))
total_bytes = sum(f.stat().st_size for f in all_txts)

print(f"Total .txt files in extracted/: {len(all_txts):,}")
print(f"Total size:                     {total_bytes / 1024 / 1024:.1f} MB")
print(f"Drive Desktop will sync new files to Google Drive within ~1 min.")
print()

sample_candidates = sorted(
    OUTPUT_DIR.glob("CSM24*_CLAIM_OF_PLAINTIFF.txt"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if sample_candidates:
    sample = sample_candidates[0]
    text = sample.read_text(encoding="utf-8")
    print(f"Spot-check: {sample.name} ({len(text):,} chars)")
    print("-" * 60)
    print(text[:800])
    print("-" * 60)
    print("Look for: form headers, plaintiff/defendant names, addresses, amounts, dates.")

Total .txt files in extracted/: 9,087
Total size:                     51.2 MB
Drive Desktop will sync new files to Google Drive within ~1 min.

Spot-check: CSM24869326_CLAIM_OF_PLAINTIFF.txt (23,290 chars)
------------------------------------------------------------
Docusign Envelope ID: FDE81985-ECD3-4F38-B1C6-832AAF3BB22A
Plaintiff's Claim and ORDER
Clerk stamps date here when form is filed
SC-100
to Go to Small Claims Court
Notice to the person being sued:
FILED
• You are the defendant if your name is listed in 2 on page 2 of this form or
Superior Court of California
County of San Francisco
on form SC-100A. The person suing you is the plaintiff, listed in
on page 2.
DEC 3 1 2024
• You and the plaintiff must go to court on the trial date listed below. If you
THE COURT
do not go to court, you may lose the case. If you lose, the court can order
BY:
that your wages, money, or property be taken to pay this claim.
Deputy Clerk
• Bring witnesses, receipts, and any evidence you need to prov